## Part 4 - Information Retrieval Using BM25

Build a BM25-based search engine over the corpus (document-level)

In [1]:
from Lab5_IR_bm25_skeleton import BM25Index

In [2]:
docs = BM25Index.load_druglib_csv("./data/druglib_outputs/processed/druglib_processed_combined.csv")
index = BM25Index(docs)

In [3]:
# test it with a sample query
results = index.score("anxiety insomnia sleep problems", top_k=10)
results

[('2564', 10.683909975799356),
 ('2928', 9.763506867390618),
 ('2489', 9.760682262861767),
 ('918', 9.425977913947644),
 ('2189', 9.330302595828465),
 ('2571', 9.273840718090973),
 ('2490', 8.75910926262448),
 ('284', 8.742203855505162),
 ('3025', 8.737655498312446),
 ('524', 8.509875060082306)]

Query Interface:
To use it in the CLI-> ```python search_drugs.py --top_k 5```

Drop into an interactive prompt in a real terminal (type queries, quit to exit); if you pipe input or redirect stdin, it falls back to running three sample queries so it's still useful non-interactively (e.g. for grading scripts or CI).

Evaluation: Precision@k, Recall@k, MAP, MRR, nDCG@k
Test queries are constructed.

In [ ]:
docs = BM25Index.load_druglib_csv("druglib_processed_combined.csv")
index = BM25Index(docs)
index.run_queries(query_file="drug_queries.txt", output_file="drug_run.txt",
                   top_k=20, run_tag="BM25-drug", base_dir="", query_format="simple")
report = index.eval(run_file="drug_run.txt", rel_file="drug_qrels.txt", k=10, base_dir="")

### Error Analysis: Query Drift, Stopword Effects, and Term Mismatch

#### Motivation

Aggregate retrieval metrics (Table 1) show strong overall performance, but two
queries — **weight loss** (AP@10 = 0.115) and **bipolar disorder**
(AP@10 = 0.177) — trail well behind the macro average (MAP@10 = 0.561). This
section investigates why, examining three candidate failure modes commonly
cited in IR literature: stopword effects, term (vocabulary) mismatch, and
query drift.

**Table 1. Macro retrieval metrics (k = 10, N = 4,142 documents, 29 queries)**

| Metric | Value |
|---|---|
| Precision@10 | 0.679 |
| Recall@10 | 0.138 |
| MAP@10 | 0.561 |
| MRR@10 | 0.869 |
| nDCG@10 | 0.698 |

#### Method

For each underperforming query, the tokenized query terms were checked
against (a) the BM25 index's stopword/tokenization pipeline, (b) the literal
review text of documents judged relevant via the `condition` metadata field,
and (c) the literal review text of the full 4,142-document corpus, to isolate
whether errors originate from over-aggressive preprocessing, vocabulary
divergence between queries and reviews, or coincidental term overlap with
off-topic documents.

#### Finding 1: Stopword Effects — Ruled Out

Both queries tokenize cleanly, with every content word preserved:

| Query | Tokenized |
|---|---|
| `weight loss` | `['weight', 'loss']` |
| `bipolar disorder` | `['bipolar', 'disorder']` |

Testing more natural phrasings confirms the stopword filter behaves as
intended elsewhere in the corpus, removing function words without harming
content terms:

| Input | Tokenized |
|---|---|
| `medication for weight loss` | `['medication', 'weight', 'loss']` |
| `treatment for high blood pressure` | `['treatment', 'high', 'blood', 'pressure']` |

**Conclusion:** stopword removal is not a contributing factor for either
query.

#### Finding 2: Term Mismatch (Vocabulary Gap)

Reviewers frequently describe conditions and outcomes in colloquial language
rather than clinical terminology, so a meaningful share of relevant documents
never contain the literal query terms at all:

**Table 2. Literal term coverage among relevant documents**

| Query | Relevant docs | Contain both query terms |
|---|---|---|
| `weight loss` | 20 | 9 (45%) |
| `bipolar disorder` | 61 | 13 (21%)|

For `weight loss`, reviewers write "dropped pounds," "curbed my appetite," or
"slimmed down" instead of the literal phrase. For `bipolar disorder`,
reviewers overwhelmingly say "bipolar," "bipolar ii," or "bi-polar" without
ever adding "disorder" — a single-term match is common, a two-term match is
not. Since BM25 (and any lexical/bag-of-words model) can only retrieve
documents containing the query's literal vocabulary, this caps the achievable
recall for these queries regardless of ranking quality.

#### Finding 3: Query Drift — the Dominant Cause

The larger effect is **query drift**: unigram BM25 has no phrase or
proximity awareness, so two terms score identically whether they appear
adjacent ("weight loss diet") or in unrelated sentences describing an
incidental side effect ("no significant weight gain... some loss of
appetite"). Both query terms in each case are common enough, across
*unrelated* conditions, to produce false positives that outrank the truly
relevant documents.

**Table 3. Term co-occurrence: relevant set vs. full corpus**

| Query terms | Relevant docs (target) | Corpus docs containing both terms |
|---|---|---|
| "weight" + "loss" | 20 | 143 |
| "bipolar" + "disorder"* | 61 | 79 total contain "disorder"; only 13 overlap with "bipolar" |

*"disorder" alone appears across conditions as varied as "attention deficit
disorder," "seizure disorder," "anxiety disorder," and "panic attacks" — a
generic diagnostic suffix rather than a bipolar-specific signal.

**Observed effect on rankings (top 10, `weight loss`):** Effexor
(anxiety/depression) and Topamax (migraine) — drugs whose reviews mention
weight change only as an incidental side effect — outrank Adipex-P, the
actual weight-loss medication, which does not appear until rank 4.

**Observed effect on rankings (top 10, `bipolar disorder`):** Effexor for
**"anxiety disorder"** ranks #2, ahead of several on-topic Lithium/Paxil
results, solely because it shares the generic term "disorder" with the
query.

With 143 candidate "drift" documents competing against only 20 truly
relevant ones for `weight loss` (and a similarly generic-term problem for
`bipolar disorder`), query drift — not stopword handling or an absolute
absence of vocabulary overlap — is the dominant cause of the low precision
and AP scores for both queries.

#### Secondary Note: Hyphenation in Tokenization

While investigating the "bi-polar disorder" condition, we observed that
`_tokenize()`'s punctuation-stripping step (`re.sub(r'[^a-zA-Z\s]', '',
text)`) deletes hyphens rather than replacing them with whitespace. In this
instance it was harmless — "bi-polar" collapses to "bipolar," coincidentally
matching the query term — but the same rule would silently merge an
unrelated hyphenated compound (e.g. "self-report" → "selfreport") into a
nonsense token that matches nothing. This did not affect the queries
analyzed here but is a general robustness gap worth addressing
(`re.sub(r'[^a-zA-Z\s]', ' ', text)` would split on hyphens instead of
deleting them).

#### Conclusions and Recommendations

1. **Stopword handling is not implicated** in either query's underperformance
   and requires no changes for these cases.
2. **Term mismatch (synonymy)** limits the achievable recall ceiling for
   colloquially-described conditions; addressing it would require query
   expansion via a medical synonym resource (e.g. UMLS) or a learned
   embedding-based retrieval stage.
3. **Query drift** is the primary driver of the low precision observed for
   both queries, and is a structural limitation of unigram bag-of-words
   scoring rather than an implementation defect. Mitigations include:
   - Phrase-aware or proximity-boosted scoring (e.g. rewarding documents
     where query terms appear within a bounded window of each other).
   - Bigram/n-gram indexing alongside unigrams for high-value query phrases.
   - A secondary re-ranking stage (e.g. cross-encoder or embedding
     similarity) applied to the BM25 top-k candidates, filtering out
     documents where the terms match without genuine topical relevance.